# 1. Setup
Load the reusable project modules and select an outcome configuration.

In [ ]:
from pathlib import Path

from postop_aki.config import load_experiment_config
from postop_aki.data import load_modeling_data, split_cohorts
from postop_aki.pipeline import evaluate_experiment, train_experiment

# Change only these paths to switch outcome or private input data.
PROJECT_ROOT = Path("..").resolve()
CONFIG_PATH = PROJECT_ROOT / "configs" / "any_aki.yaml"
DATA_PATH = PROJECT_ROOT / "outcome_full_ml.csv"
config = load_experiment_config(CONFIG_PATH)

# 2. Validate sample data
Check the fully synthetic workbook schema without fitting a model.

In [ ]:
sample = load_modeling_data(PROJECT_ROOT / "data" / "sample.xlsx")
sample_cohorts = split_cohorts(sample)
{"columns": sample.shape[1], "internal": len(sample_cohorts.internal), "external": len(sample_cohorts.external)}

# 3. Load the private dataset
Confirm the completed internal and external cohorts before modeling.

In [ ]:
data = load_modeling_data(DATA_PATH)
cohorts = split_cohorts(data)
{"internal": len(cohorts.internal), "external": len(cohorts.external)}

# 4. Train models
Fit the full, preoperative, and reduced variants with frozen settings.

In [ ]:
# Set tune=True only when intentionally repeating the five-fold search.
trained_models = train_experiment(
    DATA_PATH, config, PROJECT_ROOT / "outputs" / "models", tune=False
)
trained_models

# 5. Evaluate models
Calculate internal and external metrics with separate Youden cutoffs.

In [ ]:
# Omit model_dir to evaluate the checksum-verified frozen artifacts.
results = evaluate_experiment(DATA_PATH, config, n_bootstrap=2_000)

# 6. Review results
Display the concise aggregate performance table returned by the pipeline.

In [ ]:
results.round(3)